In [1]:
###############################################################################
# The Institute for the Design of Advanced Energy Systems Integrated Platform
# Framework (IDAES IP) was produced under the DOE Institute for the
# Design of Advanced Energy Systems (IDAES).
#
# Copyright (c) 2018-2026 by the software owners: The Regents of the
# University of California, through Lawrence Berkeley National Laboratory,
# National Technology & Engineering Solutions of Sandia, LLC, Carnegie Mellon
# University, West Virginia University Research Corporation, et al.
# All rights reserved.  Please see the files COPYRIGHT.md and LICENSE.md
# for full copyright and license information.
###############################################################################

# Using Parameter Estimation with Modular Property Packages
Author: Alejandro Garciadego  
Maintainer: Stephen Cini  
Updated: 2026-06-11  
## 1. Introduction

This Jupyter Notebook estimates binary interaction parameters for a CO$_2$-Ionic liquid property package. A property package has been created for CO$_2$-[bmim][PF6]. We will utilize Pyomo's `parmest` tool in conjunction with IDAES models for parameter estimation. We demonstrate these tools by estimating the parameters associated with the Peng-Robinson property model for a benzene-toluene mixture. The Peng-Robinson EOS the binary interaction parameter (kappa_ij). When estimating parameters associated with the property package, IDAES provides the flexibility of doing the parameter estimation by just using the state block or by using a unit model with a specified property package. This module will demonstrate parameter estimation by using the flash unit model with a Modular Property Package.

### 1.1 Tutorial objectives

* Utilize the Modular Property Package framework, which provides a flexible platform for users to build property packages by calling upon libraries of modular sub-models to build up complex property calculations with the least effort possible.
* Set up a method to return an initialized model
* Set up the parameter estimation problem using `parmest`

## 2. Problem Statement

### 2.1 Importing Pyomo and IDAES model and flowsheet components.

In the next cell, we will be importing the necessary components from Pyomo and IDAES.

In [2]:
# Import objects from pyomo package
from pyomo.environ import ConcreteModel, SolverFactory, value, Suffix

# Import the main FlowsheetBlock from IDAES. The flowsheet block will contain the unit model
from idaes.core import FlowsheetBlock

# Import idaes logger to set output levels
import idaes.logger as idaeslog

# Import Flash unit model from idaes.models.unit_models
from idaes.models.unit_models import Flash

### 2.2 Import parmest 

In [3]:
import pyomo.contrib.parmest.parmest as parmest

### 2.3 Import the Modular Property framework

In [4]:
from idaes.models.properties.modular_properties.examples.CO2_bmimPF6_PR import (
    configuration,
)

from idaes.models.properties.modular_properties import GenericParameterBlock

### 2.4 Import data

In the next cell, we will be importing `pandas` and the `.csv` file with preassure and composition data. For this example, we load data from the csv file CO2_IL_298.csv. The dataset consists of ninteen data points which provide the mole fraction of [bmim][PF6] and carbon dioxide and the pressure at three different temperatures.

In [5]:
import pandas as pd

# Load data from csv
data = pd.read_csv("CO2_IL_298.csv")

## 3.0 Setting up an initialized model

We need to provide a method that returns an initialized model to the `parmest` tool in Pyomo.

How we build the model will depend on the data we provided in the data dataframe from pir .csv file.

In this case we have data on the liquid mixture, the temperature and the pressure. We will fix the temperature, mole franction in the liquid phase, and the mole fraction of the inlet. 

In [6]:
def PR_model(data):

    m = ConcreteModel()

    m.fs = FlowsheetBlock(dynamic=False)

    m.fs.properties = GenericParameterBlock(**configuration)

    m.fs.state_block = m.fs.properties.build_state_block([1], defined_state=True)

    m.fs.state_block[1].flow_mol.fix(1)
    if isinstance(data, dict) or isinstance(data, pd.Series):
        x = float(data["x_carbon_dioxide"]) + 0.5
        m.fs.state_block[1].temperature.fix(float(data["temperature"]))
        m.fs.state_block[1].pressure.fix(float(data["pressure"]))
    elif isinstance(data, pd.DataFrame):
        x = float(data.iloc[0]["x_carbon_dioxide"]) + 0.5
        m.fs.state_block[1].temperature.fix(float(data.iloc[0]["temperature"]))
        m.fs.state_block[1].pressure.fix(float(data.iloc[0]["pressure"]))
    else:
        raise ValueError("Unrecognized data type.")
    m.fs.state_block[1].mole_frac_comp["bmimPF6"].fix(1 - x)
    m.fs.state_block[1].mole_frac_comp["carbon_dioxide"].fix(x)

    # parameter - kappa_ij (set at 0.3, 0 if i=j)
    m.fs.properties.PR_kappa["bmimPF6", "bmimPF6"].fix(0)
    m.fs.properties.PR_kappa["bmimPF6", "carbon_dioxide"].fix(-0.047)
    m.fs.properties.PR_kappa["carbon_dioxide", "carbon_dioxide"].fix(0)
    m.fs.properties.PR_kappa["carbon_dioxide", "bmimPF6"].fix(0.002)

    # Initialize the flash unit
    m.fs.state_block.initialize(outlvl=idaeslog.INFO)

    # Fix the state variables on the state block
    if isinstance(data, dict) or isinstance(data, pd.Series):
        m.fs.state_block[1].temperature.fix(float(data["temperature"]))
        m.fs.state_block[1].mole_frac_phase_comp["Liq", "bmimPF6"].fix(
            float(data["x_bmimPF6"])
        )
        m.fs.state_block[1].mole_frac_phase_comp["Liq", "carbon_dioxide"].fix(
            float(data["x_carbon_dioxide"])
        )
        m.fs.state_block[1].mole_frac_comp["bmimPF6"].fix(float(data["x_bmimPF6"]))
    elif isinstance(data, pd.DataFrame):
        m.fs.state_block[1].temperature.fix(float(data.iloc[0]["temperature"]))
        m.fs.state_block[1].mole_frac_phase_comp["Liq", "bmimPF6"].fix(
            float(data.iloc[0]["x_bmimPF6"])
        )
        m.fs.state_block[1].mole_frac_phase_comp["Liq", "carbon_dioxide"].fix(
            float(data.iloc[0]["x_carbon_dioxide"])
        )
        m.fs.state_block[1].mole_frac_comp["bmimPF6"].fix(
            float(data.iloc[0]["x_bmimPF6"])
        )
    else:
        raise ValueError("Unrecognized data type.")

    m.fs.state_block[1].pressure.unfix()
    m.fs.state_block[1].mole_frac_comp["carbon_dioxide"].unfix()
    # Set bounds on variables to be estimated
    m.fs.properties.PR_kappa["bmimPF6", "carbon_dioxide"].setlb(-5)
    m.fs.properties.PR_kappa["bmimPF6", "carbon_dioxide"].setub(5)

    m.fs.properties.PR_kappa["carbon_dioxide", "bmimPF6"].setlb(-5)
    m.fs.properties.PR_kappa["carbon_dioxide", "bmimPF6"].setub(5)

    # Return initialized flash model
    return m

### 3.1 Solving square problem

In [7]:
from idaes.core.util.model_statistics import degrees_of_freedom
import pytest

test_data = {
    "temperature": 298,
    "pressure": 812323,
    "x_bmimPF6": 0.86,
    "x_carbon_dioxide": 0.14,
}

m = PR_model(test_data)

# Check that degrees of freedom is 0
assert degrees_of_freedom(m) == 0

# Solve the model with the default solver and display results
solver = SolverFactory("ipopt")
results = solver.solve(m, tee=True)

2026-06-11 12:42:22 [INFO] idaes.init.fs.state_block: Starting initialization
2026-06-11 12:42:23 [INFO] idaes.init.fs.state_block: Bubble, dew, and critical point initialization: optimal - Optimal Solution Found.
2026-06-11 12:42:23 [INFO] idaes.init.fs.state_block: Equilibrium temperature initialization completed.
2026-06-11 12:42:23 [INFO] idaes.init.fs.state_block: State variable initialization completed.
2026-06-11 12:42:23 [INFO] idaes.init.fs.state_block: Phase equilibrium initialization: optimal - Optimal Solution Found.
2026-06-11 12:42:23 [INFO] idaes.init.fs.state_block: Property initialization: optimal - Optimal Solution Found.
2026-06-11 12:42:23 [INFO] idaes.init.fs.state_block: Property package initialization: optimal - Optimal Solution Found.
Ipopt 3.13.2: 

******************************************************************************
This program contains Ipopt, a library for large-scale nonlinear optimization.
 Ipopt is released as open source code under the Eclipse 

## 4.0 Parameter estimation using parmest 

### 4.1 Define the Experiment class

Define the Experiment class to label model for parameter estimation.

In [8]:
from pyomo.contrib.parmest.experiment import Experiment


# Create experiment class for parameter estimation
class PRExperiment(Experiment):

    def __init__(self, data, meas_error=None):
        """Initialize the PR Experiment class

        Args:
            data: DataFrame containing the experimental data
            meas_error: Measurement error for the data (optional)
        """
        self.model = None
        self.data = data
        self.meas_error = meas_error

    def create_model(self):
        """Create the Pyomo model for the PR parameter estimation problem"""
        self.model = PR_model(self.data)

    def label_model(self):
        m = self.model

        # Add Suffixes to label the outputs, parameters, and measurement error in the model
        m.experiment_outputs = Suffix(direction=Suffix.LOCAL)
        m.experiment_outputs.update(
            [(m.fs.state_block[1].pressure, self.data["pressure"])]
        )

        m.measurement_error = Suffix(direction=Suffix.LOCAL)
        m.measurement_error.update([(m.fs.state_block[1].pressure, self.meas_error)])

        # Add unknown parameters to the model for easier access
        m.unknown_parameters = Suffix(direction=Suffix.LOCAL)
        m.unknown_parameters.update(
            (k, value(k))
            for k in [
                m.fs.properties.PR_kappa["bmimPF6", "carbon_dioxide"],
                m.fs.properties.PR_kappa["carbon_dioxide", "bmimPF6"],
            ]
        )

    def get_labeled_model(self):
        """Return the labeled model"""
        if self.model is None:
            self.create_model()
            self.label_model()
        return self.model

### 4.2 Pre-process the data into individual experiments

We now separate our data and assign a model for each individual experiments, creating an experiment list. 

In [9]:
# Update to new interface
exp_list = []
for i in range(data.shape[0]):
    exp_list.append(PRExperiment(data.iloc[i]))

### 4.3 Run the parameter estimation

We are now ready to set up the parameter estimation problem. We will create a parameter estimation object called pest. As shown below, we pass the experiment list, and an objective function to the Estimator method. tee=True will print the solver output after solving the parameter estimation problem.

In [10]:
pest = parmest.Estimator(exp_list, obj_function="SSE", tee=True)
obj_value, parameters = pest.theta_est()

2026-06-11 12:42:23 [INFO] idaes.init.fs.state_block: Starting initialization
2026-06-11 12:42:23 [INFO] idaes.init.fs.state_block: Bubble, dew, and critical point initialization: optimal - Optimal Solution Found.
2026-06-11 12:42:23 [INFO] idaes.init.fs.state_block: Equilibrium temperature initialization completed.
2026-06-11 12:42:23 [INFO] idaes.init.fs.state_block: State variable initialization completed.
2026-06-11 12:42:23 [INFO] idaes.init.fs.state_block: Phase equilibrium initialization: optimal - Optimal Solution Found.
2026-06-11 12:42:23 [INFO] idaes.init.fs.state_block: Property initialization: optimal - Optimal Solution Found.
2026-06-11 12:42:23 [INFO] idaes.init.fs.state_block: Property package initialization: optimal - Optimal Solution Found.
2026-06-11 12:42:23 [INFO] idaes.init.fs.state_block: Starting initialization
2026-06-11 12:42:23 [INFO] idaes.init.fs.state_block: Bubble, dew, and critical point initialization: optimal - Optimal Solution Found.
2026-06-11 12:42:

## 5.0 Display results

Let us display the results by running the next cell.

In [11]:
print(f"The SSE at the optimal solution is {obj_value*1e-7:0.6f}")
print()
print("The values for the parameters are as follows:")
for k, v in parameters.items():
    print(f"{k} = {v}")

The SSE at the optimal solution is 29.282891

The values for the parameters are as follows:
fs.properties.PR_kappa[bmimPF6,carbon_dioxide] = -0.40714284008565715
fs.properties.PR_kappa[carbon_dioxide,bmimPF6] = 0.02059368400143062


Now we can use this parameters and include them in the configuration dictionary. We can also use `m.fs.properties = GenericParameterBlock(**configuration)` to solve unit models.